# Imports

In [1]:
import pickle

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.model_selection import StratifiedKFold, cross_val_predict

## Utils

In [2]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Datasets

In [3]:
X_train = pd.read_parquet('../data/processed/X_train_raw.parquet')
y_train = pd.read_parquet('../data/interim/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_raw.parquet')

In [4]:
X_train.head()

,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
id,,,,,,,,,,,,,
0,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female
1,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other
2,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male
3,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female
4,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,missing,average,sedentary,missing,male


In [5]:
X_test.head()

,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
id,,,,,,,,,,,,,
690088,5.35,64.9,23.48,2745.0,14167.0,59.5,1.86,veg,high,poor,active,occasional,male
690089,NaN,83.1,22.42,1773.0,6801.0,24.5,2.40,balanced,high,poor,sedentary,yes,other
690090,6.68,59.7,24.14,3040.0,13250.0,48.5,2.76,balanced,medium,poor,active,no,missing
690091,7.13,78.5,26.26,2494.0,6331.0,56.9,2.34,veg,low,good,moderate,yes,other
690092,5.49,77.7,23.29,1828.0,13894.0,39.4,2.45,veg,high,average,active,occasional,other


In [6]:
cat_features_raw = X_train.select_dtypes(include=['category', 'object']).columns.tolist()

X_train = X_train.astype({feat: 'category' for feat in cat_features_raw})
X_test = X_test.astype({feat: 'category' for feat in cat_features_raw})

cat_features_raw

['diet_type',
 'stress_level',
 'sleep_quality',
 'physical_activity_level',
 'smoking_alcohol',
 'gender']

# Machine Learning

In [7]:
models = dict(
    lgbm=load_pickle('../models/layer_1/model_lightgbm_n_trial_60.pkl'),
    xgb=load_pickle('../models/layer_1/model_xgboost_n_trial_60.pkl'),
    # cat=load_pickle('../models/layer_1/model_catboost.pkl'),
)

## Train Dataset

In [8]:
X_train_stacking = pd.DataFrame({})

In [9]:
for model_name, model in tqdm(models.items()):
    
    print(f"Predicting Train Dataset {model_name}")

    predictions = cross_val_predict(
        model, 
        X_train, 
        y_train.health_condition,
        method='predict_proba', 
        cv=StratifiedKFold(shuffle=True, random_state=42, n_splits=3),
        params={'cat_features': cat_features_raw} if model_name == 'cat' else None,
        n_jobs=1 if model_name == 'cat' else 1
    )
    
    X_train_stacking[[f'{model_name}_0', f'{model_name}_1', f'{model_name}_2']] = predictions

  0%|                                                                                                                                                                                        | 0/2 [00:00<?, ?it/s]

Predicting Train Dataset lgbm


 50%|███████████████████████████████████████████████████████████████████████████████████████▌                                                                                       | 1/2 [03:10<03:10, 190.11s/it]

Predicting Train Dataset xgb


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [04:58<00:00, 149.33s/it]


## Test Dataset

In [10]:
X_test_stacking = pd.DataFrame({})

In [11]:
for model_name, model in models.items():
    
    print(f"Predicting Test Dataset {model_name}")
    
    X_test_stacking[[f'{model_name}_0', f'{model_name}_1', f'{model_name}_2']] = model.predict_proba(X_test)

Predicting Test Dataset lgbm
Predicting Test Dataset xgb


# Saving

In [12]:
X_train_stacking.to_parquet('../data/processed/X_train_stacking_layer_one_model_60_trials.parquet')
X_test_stacking.to_parquet('../data/processed/X_test_stacking_layer_one_model_60_trials.parquet')

In [13]:
X_train_stacking.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2
0,0.004922,0.001773,0.993304,0.004134,0.002593,0.993273
1,0.997228,0.000463,0.002309,0.995591,0.000405,0.004004
2,0.002369,0.000604,0.997026,0.001958,0.000503,0.997538
3,0.004656,0.003362,0.991982,0.002466,0.002391,0.995142
4,0.995353,0.000016,0.004632,0.994178,0.000003,0.005819


In [14]:
X_test_stacking.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2
0,0.012452,0.003199,0.984349,0.011048,0.004491,0.984461
1,0.472116,0.002331,0.525553,0.499186,0.001896,0.498918
2,0.998690,0.001012,0.000298,0.997826,0.001594,0.000579
3,0.980854,0.000018,0.019128,0.987434,0.000009,0.012557
4,0.006765,0.001847,0.991388,0.006876,0.001972,0.991152


In [15]:
X_train.shape

(690088, 13)

In [16]:
X_test.shape

(295753, 13)

In [17]:
y_train.head()

,health_condition
id,
0,2
1,0
2,2
3,2
4,0
